# Week 3 - Day 2 - Hands On Lab
Predicting a Continuous Value (Linear Regression)

## Objective

Yesterday I cleaned the Auto MPG dataset and split it into training and testing sets. Today I'm training my first supervised learning model — Linear Regression — to predict fuel efficiency (`mpg`) from a car's specs. My goal isn't just to get a working model, but to understand what it actually learned (which feature matters most, according to its coefficients), and to check honestly whether it performs better than a simple baseline.

---

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---

## Load Data

I'm using the same Auto MPG dataset from yesterday, since it already has a continuous target (`mpg`) I can predict with linear regression.

In [2]:
df = pd.read_csv("data/auto-mpg.csv")
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


---

## Prepare the Data

Before training, I clean the data the same way I did yesterday: convert `horsepower` to numeric and drop the rows where it was missing (6 rows), then drop `car name` since it's text, not a usable feature. I then recreate the same 80/20 train/test split from yesterday, using the same `random_state` so the split is identical.

In [3]:
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")
df = df.dropna(subset=["horsepower"])
df = df.drop("car name", axis=1)

X = df.drop("mpg", axis=1)
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(313, 7) (79, 7) (313,) (79,)


---

## Step 1: Train the Model

Now I train a Linear Regression model on the training data. The model looks at `X_train` and `y_train` together and searches for the weights and bias that minimize the total squared error — this is the least squares process from my learning notebook, and it's exactly what `.fit()` does internally.

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

comparison = pd.DataFrame({"actual": y_test.values[:5], "predicted": predictions[:5]})
comparison

,actual,predicted
0,26.0,25.841562
1,21.6,26.036745
2,36.1,34.506018
3,26.0,24.895532
4,27.0,28.425987


---

## Step 2: Report the Coefficients

After training, I can see what the model actually learned by reading its coefficients (one weight per feature) and its intercept.

In [5]:
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef:.4f}")
print("intercept:", model.intercept_)

cylinders: -0.3458
displacement: 0.0151
horsepower: -0.0213
weight: -0.0061
acceleration: 0.0380
model year: 0.7677
origin: 1.6135
intercept: -18.499361128724747


**Interpretation:** `origin` has the largest raw coefficient (1.61), followed by `model year` (0.77). `model year` being positive makes sense — newer cars tend to be more fuel-efficient. `weight` being negative (-0.006) also makes sense — heavier cars use more fuel.

I'm cautious about calling `origin` "the most important feature" just because its coefficient is the largest. `origin` only takes values 1, 2, or 3, while `weight` ranges from about 1600 to 5000. A feature's raw coefficient size depends on its scale, not only on its real-world importance — comparing them fairly would require scaling the features first.

---

## Step 3: Evaluate the Model

To know how good these predictions actually are, I calculate three metrics on the test set: MAE, RMSE, and R².

In [6]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

MAE: 2.420
RMSE: 3.273
R2: 0.790


**Interpretation:** on average, my predictions are off by about 2.4 mpg (MAE). RMSE (3.27) is a bit higher since it penalizes larger mistakes more. R² = 0.79 means the model explains about 79% of the variation in mpg using these 7 features.

---

## Step 4: Compare Against a Baseline

These numbers alone don't tell me if the model is actually good. To find out, I compare it against the simplest possible baseline: predicting the average `mpg` for every single car, ignoring all features.

In [7]:
baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print(f"Baseline RMSE: {baseline_rmse:.3f}")
print(f"Model RMSE:    {rmse:.3f}")
print("Model beats baseline:", rmse < baseline_rmse)

Baseline RMSE: 7.185
Model RMSE:    3.273
Model beats baseline: True


**Interpretation:** the baseline's RMSE (7.19) is more than double the model's RMSE (3.27). This confirms the model has genuinely learned something useful from the features — it isn't just getting lucky, and it clearly beats guessing the average every time.

---

## Step 5: My Interpretation

My Linear Regression model predicts `mpg` with an average error of about 2.4 mpg (MAE), and explains roughly 79% of the variation in fuel efficiency (R² = 0.79) using the 7 available features.

Comparing against the baseline confirms this is a real result and not chance: the baseline's RMSE (7.19) is more than double the model's RMSE (3.27), meaning the model is genuinely learning from the features rather than just guessing the average every time.

Looking at the coefficients, `model year` has a positive effect (+0.77) — newer cars tend to be more fuel-efficient — and `weight` has a negative effect (-0.006) — heavier cars use more fuel. Both match real-world expectations. `origin` has the largest raw coefficient (1.61), but I'm cautious about calling it "the most important feature" just from that: `origin` only takes values 1, 2, or 3, while `weight` ranges from about 1600 to 5000. A feature's raw coefficient size depends on its scale, not only its real-world importance, so a fair comparison would need the features to be scaled first.

Overall, this model is a solid first attempt: it clearly beats a naive baseline, and its learned coefficients make physical sense, but comparing feature importance directly from unscaled coefficients would be misleading.

---

## Summary

In this lab, I trained my first supervised learning model — Linear Regression — on the Auto MPG dataset, predicting `mpg` from 7 car features. I evaluated it using MAE, RMSE, and R², and confirmed it meaningfully outperforms a baseline that just predicts the average. I also learned that comparing coefficients directly across features only gives a fair sense of importance when those features are on the same scale.